In [ ]:
# ================================
# STEP 1: DATA INGESTION
# ================================

# Import required library
import pandas as pd

# -------------------------------
# 1. Define file paths
# -------------------------------
# These paths point to your dataset files
customers_path = "C:/Users/91985/OneDrive/Documents/Infosys Internship/AI-Based-Knowledge-Graph-Builder-for-Enterprise-Intelligence/milestone_3/data/dim_customer.csv"

products_path = "C:/Users/91985/OneDrive/Documents/Infosys Internship/AI-Based-Knowledge-Graph-Builder-for-Enterprise-Intelligence/milestone_3/data/dim_product.csv"

orders_path = "C:/Users/91985/OneDrive/Documents/Infosys Internship/AI-Based-Knowledge-Graph-Builder-for-Enterprise-Intelligence/milestone_3/data/fact_order.csv"
# -------------------------------
# 2. Load datasets
# -------------------------------
# Read CSV files into pandas DataFrames
customers = pd.read_csv(customers_path)
products = pd.read_csv(products_path)
orders = pd.read_csv(orders_path)

# -------------------------------
# 3. Quick inspection
# -------------------------------
# Display first few rows to understand structure

print("===== Customers Table =====")
display(customers.head())

print("\n===== Products Table =====")
display(products.head())

print("\n===== Orders Table =====")
display(orders.head())

# -------------------------------
# 4. Check basic info (optional but useful)
# -------------------------------
# Helps you understand columns and data types

print("\n===== Data Info =====")
print("Customers:")
print(customers.info())

print("\nProducts:")
print(products.info())

print("\nOrders:")
print(orders.info())

===== Customers Table =====


,customer_id,customer_name,region,age,gender,Customer_since
0,1,Robin Ward,North,51,Male,2022-07-26
1,2,Joseph Chapman,East,21,Female,2023-04-21
2,3,Dr. Alicia Henderson DVM,North,19,Male,2024-03-22
3,4,Laura King,Central,20,Male,2024-08-03
4,5,Brian Wheeler,South,18,Female,2023-02-07



===== Products Table =====


,product_id,product_name,category,unit_price,supplier
0,1001,Huge,Apparel,99.32,Owens Inc
1,1002,Address,Furniture,290.67,Romero and Sons
2,1003,Almost,Electronics,37.75,"Williams, Shah and Mitchell"
3,1004,Month,Beauty,75.32,Williams-Stephens
4,1005,Plan,Beauty,332.12,Gibson Inc



===== Orders Table =====


,order_id,order_date,customer_id,product_id,quantity,unit_price,sales_amount,order_year,order_month,quantity_norm,sales_amount_norm
0,1,2022-11-21,394,1005,2,332.12,664.24,2022,2022-11,0.125,0.145499
1,296,2023-09-11,271,1005,3,332.12,996.36,2023,2023-09,0.250,0.221177
2,549,2023-06-09,190,1005,4,332.12,1328.48,2023,2023-06,0.375,0.296854
3,606,2023-10-28,280,1005,6,332.12,1992.72,2023,2023-10,0.625,0.448209
4,776,2024-07-05,308,1005,7,332.12,2324.84,2024,2024-07,0.750,0.523887



===== Data Info =====
Customers:
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   customer_id     1000 non-null   int64
 1   customer_name   1000 non-null   str  
 2   region          1000 non-null   str  
 3   age             1000 non-null   int64
 4   gender          1000 non-null   str  
 5   Customer_since  1000 non-null   str  
dtypes: int64(2), str(4)
memory usage: 47.0 KB
None

Products:
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    100 non-null    int64  
 1   product_name  100 non-null    str    
 2   category      100 non-null    str    
 3   unit_price    100 non-null    float64
 4   supplier      100 non-null    str    
dtypes: float64(1), int64(1), str(3)
memory usage: 4.0 KB
None

Orders:


In [ ]:
# Merge Tables

# Join orders → customers → products
merged = orders.merge(customers, on="customer_id", how="left")
merged = merged.merge(products,  on="product_id",  how="left")

print("Merged shape:", merged.shape)
display(merged.head(3))

Merged shape: (2000, 20)


,order_id,order_date,customer_id,product_id,quantity,unit_price_x,sales_amount,order_year,order_month,quantity_norm,sales_amount_norm,customer_name,region,age,gender,Customer_since,product_name,category,unit_price_y,supplier
0,1,2022-11-21,394,1005,2,332.12,664.24,2022,2022-11,0.125,0.145499,Lori Roberts,East,35,Female,2020-01-09,Plan,Beauty,332.12,Gibson Inc
1,296,2023-09-11,271,1005,3,332.12,996.36,2023,2023-09,0.250,0.221177,Katherine Bowman,North,37,Female,2020-08-21,Plan,Beauty,332.12,Gibson Inc
2,549,2023-06-09,190,1005,4,332.12,1328.48,2023,2023-06,0.375,0.296854,Jessica Flores,East,60,Male,2020-07-01,Plan,Beauty,332.12,Gibson Inc


In [ ]:
# Convert Rows to Text Documents

def to_text(row):
    return (
        f"Customer {row['customer_name']} from {row['region']} "
        f"purchased {row['product_name']} (category: {row['category']}) "
        f"in quantity {row['quantity']} "
        f"for total sales of {row['sales_amount']}."
    )

documents = [to_text(row) for _, row in merged.iterrows()]

print(f"Total documents: {len(documents)}")
print("\nSample:")
print(documents[0])


Total documents: 2000

Sample:
Customer Lori Roberts from East purchased Plan (category: Beauty) in quantity 2 for total sales of 664.24.


In [ ]:
# Connection

from groq import Groq


GROQ_API_KEY = "#"  

client = Groq(api_key=GROQ_API_KEY)

# Quick test
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",   
    messages=[
        {"role": "user", "content": "Reply with only the word: OK"}
    ]
)

print(response.choices[0].message.content)

OK


In [ ]:
# Extract Entities & Relationships

import json
import re

def extract_knowledge(text):
    
    prompt = f"""You are an information extraction system.
Extract entities and relationships from the text below.

Text: "{text}"

Return ONLY this JSON structure, nothing else:
{{
  "entities": [
    {{"name": "value", "type": "Customer|Product|Region|Category"}}
  ],
  "relationships": [
    {{"source": "value", "relation": "VERB", "target": "value"}}
  ]
}}

Rules:
- Use ONLY values from the text
- Entity types must be exactly: Customer, Product, Region, or Category
- Relation must be uppercase verb like: PURCHASED, LOCATED_IN, BELONGS_TO
- Return valid JSON only, no explanation"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",   
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )
    
    return response.choices[0].message.content


def clean_output(raw):
    try:
        return json.loads(raw)
    except:
        try:
            start = raw.find("{")
            end   = raw.rfind("}") + 1
            return json.loads(raw[start:end])
        except:
            print(" Could not parse:", raw[:100])
            return None


def normalize(data):
    if data is None:
        return {"entities": [], "relationships": []}
    
    entities = [
        {"name": e["name"], "type": e["type"]}
        for e in data.get("entities", [])
        if "name" in e and "type" in e
    ]
    
    relationships = [
        {
            "source":   r["source"],
            "relation": r["relation"].upper(),
            "target":   r["target"]
        }
        for r in data.get("relationships", [])
        if all(k in r for k in ["source", "relation", "target"])
    ]
    
    return {"entities": entities, "relationships": relationships}

In [ ]:
#  Document Test

sample = documents[0]
print("Input:")
print(sample)
print("\n--- Raw output ---")

raw    = extract_knowledge(sample)
print(raw)

print("\n--- Cleaned output ---")
parsed = clean_output(raw)
result = normalize(parsed)
print(result)

Input:
Customer Lori Roberts from East purchased Plan (category: Beauty) in quantity 2 for total sales of 664.24.

--- Raw output ---
{
  "entities": [
    {"name": "Lori Roberts", "type": "Customer"},
    {"name": "East", "type": "Region"},
    {"name": "Plan", "type": "Product"},
    {"name": "Beauty", "type": "Category"}
  ],
  "relationships": [
    {"source": "Lori Roberts", "relation": "PURCHASED", "target": "Plan"},
    {"source": "Lori Roberts", "relation": "LOCATED_IN", "target": "East"},
    {"source": "Plan", "relation": "BELONGS_TO", "target": "Beauty"}
  ]
}

--- Cleaned output ---
{'entities': [{'name': 'Lori Roberts', 'type': 'Customer'}, {'name': 'East', 'type': 'Region'}, {'name': 'Plan', 'type': 'Product'}, {'name': 'Beauty', 'type': 'Category'}], 'relationships': [{'source': 'Lori Roberts', 'relation': 'PURCHASED', 'target': 'Plan'}, {'source': 'Lori Roberts', 'relation': 'LOCATED_IN', 'target': 'East'}, {'source': 'Plan', 'relation': 'BELONGS_TO', 'target': 'Beauty'

In [ ]:
# Take only first 300 documents
documents = documents[:300]
print(f"Working with {len(documents)} documents")

Working with 300 documents


In [ ]:
# Process All Documents

import time
import json

BATCH_SIZE = 50
SAVE_FILE  = "results.json"
all_results = []

total = len(documents)

for i, doc in enumerate(documents):
    
    try:
        raw    = extract_knowledge(doc)
        parsed = clean_output(raw)
        result = normalize(parsed)
        
        all_results.append({
            "document":      doc,
            "entities":      result["entities"],
            "relationships": result["relationships"]
        })
    
    except Exception as e:
        print(f" Error on document {i}: {e}")
        all_results.append({
            "document":      doc,
            "entities":      [],
            "relationships": [] 
        })
    
    # Save every 50 documents
    if (i + 1) % BATCH_SIZE == 0:
        with open(SAVE_FILE, "w") as f:
            json.dump(all_results, f)
        print(f"✅ Saved {i+1}/{total}")
    
    time.sleep(0.5)

# Final save
with open(SAVE_FILE, "w") as f:
    json.dump(all_results, f)

print(f"\n Done! Total processed: {len(all_results)}")


In [ ]:
import json

with open("results.json", "r") as f:
    all_results = json.load(f)

# Count valid results (non-empty)
valid   = [r for r in all_results if len(r["entities"]) > 0]
empty   = [r for r in all_results if len(r["entities"]) == 0]

total_entities      = sum(len(r["entities"])      for r in valid)
total_relationships = sum(len(r["relationships"]) for r in valid)

print(f"Total saved       : {len(all_results)}")
print(f"Valid results     : {len(valid)}")
print(f"Empty (API errors): {len(empty)}")
print(f"Total entities    : {total_entities}")
print(f"Total relationships: {total_relationships}")

Total saved       : 300
Valid results     : 184
Empty (API errors): 116
Total entities    : 736
Total relationships: 552


In [ ]:
# Connecting to Neo4J

from neo4j import GraphDatabase

NEO4J_URI      = "x"  
NEO4J_USERNAME = "x"
NEO4J_PASSWORD = "x"                          

# Close old driver if exists
try:
    driver.close()
except:
    pass

# Fresh connection
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

with driver.session() as session:
    result = session.run("RETURN 1 AS test")
    print("✅ Connected! Result:", result.single()["test"])

✅ Connected! Result: 1


In [ ]:
#insert fun  

def insert_to_neo4j(session, entities, relationships):
    
    for entity in entities:
        session.run(
            "MERGE (n {name: $name}) SET n.type = $type",
            name=entity["name"],
            type=entity["type"]
        )
    
    for rel in relationships:
        session.run(
            """
            MATCH (a {name: $source})
            MATCH (b {name: $target})
            MERGE (a)-[r:RELATED {type: $relation}]->(b)
            """,
            source=rel["source"],
            target=rel["target"],
            relation=rel["relation"]
        )

In [ ]:
# Insert All Data

valid_results = [r for r in all_results if len(r["entities"]) > 0]
print(f"Inserting {len(valid_results)} records into Neo4j...")

with driver.session() as session:
    for i, result in enumerate(valid_results):
        insert_to_neo4j(
            session,
            result["entities"],
            result["relationships"]
        )
        if (i + 1) % 20 == 0:
            print(f"✅ Inserted {i+1}/{len(valid_results)}")

print("\n All data inserted into Neo4j!")

Inserting 184 records into Neo4j...
✅ Inserted 20/184
✅ Inserted 40/184
✅ Inserted 60/184
✅ Inserted 80/184
✅ Inserted 100/184
✅ Inserted 120/184
✅ Inserted 140/184
✅ Inserted 160/184
✅ Inserted 180/184

🎉 All data inserted into Neo4j!


In [ ]:
# Verify

with driver.session() as session:
    
    nodes = session.run("MATCH (n) RETURN count(n) as count")
    print(f"Total nodes        : {nodes.single()['count']}")
    
    rels = session.run("MATCH ()-[r]->() RETURN count(r) as count")
    print(f"Total relationships: {rels.single()['count']}")
    
    print("\nNodes by type:")
    result = session.run("""
        MATCH (n)
        RETURN n.type as type, count(n) as count
        ORDER BY count DESC
    """)
    for record in result:
        print(f"  {record['type']}: {record['count']}")

Total nodes        : 182
Total relationships: 353

Nodes by type:
  Customer: 163
  Product: 9
  Region: 5
  Category: 5


In [ ]:
# Load Embedding Model

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

test = model.encode("Customer bought Beauty products")
print(f" Embedding model loaded!")
print(f"Embedding size: {len(test)}")

In [ ]:
# Build FAISS Index

import faiss
import numpy as np

texts = [r["document"] for r in all_results if len(r["entities"]) > 0]
print(f"Encoding {len(texts)} documents...")

embeddings   = model.encode(texts, show_progress_bar=True)
dimension    = embeddings.shape[1]
index_faiss  = faiss.IndexFlatL2(dimension)
index_faiss.add(np.array(embeddings))

print(f"✅ FAISS index built!")
print(f"Total vectors: {index_faiss.ntotal}")

Encoding 184 documents...


Batches: 100%|██████████| 6/6 [00:02<00:00,  2.44it/s]

✅ FAISS index built!
Total vectors: 184


In [ ]:
# Test Semantic Search

def semantic_search(query, top_k=5):
    query_vector = model.encode([query])
    distances, indices = index_faiss.search(np.array(query_vector), top_k)
    
    print(f"Query: '{query}'")
    print(f"\nTop {top_k} results:\n")
    for i, idx in enumerate(indices[0]):
        print(f"{i+1}. {texts[idx]}")
        print(f"   Score: {distances[0][i]:.4f}\n")

# Test it
semantic_search("customers who buy beauty products")

Query: 'customers who buy beauty products'

Top 5 results:

1. Customer Alejandra Walton from North purchased Free (category: Beauty) in quantity 2 for total sales of 732.74.
   Score: 0.9262

2. Customer Christina Stewart from Central purchased Free (category: Beauty) in quantity 5 for total sales of 1831.85.
   Score: 0.9311

3. Customer Sandra Lane from East purchased Free (category: Beauty) in quantity 9 for total sales of 3297.33.
   Score: 0.9326

4. Customer Angela Watson from North purchased Plan (category: Beauty) in quantity 5 for total sales of 1660.6.
   Score: 0.9420

5. Customer Rachel Leon from North purchased Free (category: Beauty) in quantity 7 for total sales of 2564.59.
   Score: 0.9498



In [ ]:

#  Build BM25 Index

from rank_bm25 import BM25Okapi

tokenized = [doc.lower().split() for doc in texts]
bm25      = BM25Okapi(tokenized)

print(f"✅ BM25 keyword index built!")
print(f"Total documents indexed: {len(tokenized)}")

✅ BM25 keyword index built!
Total documents indexed: 184


In [ ]:
#  Hybrid Search (Keyword + Semantic Combined)

def hybrid_search(query, top_k=5):
    
    # --- Semantic results ---
    query_vector       = model.encode([query])
    distances, indices = index_faiss.search(np.array(query_vector), top_k)
    semantic_idx       = list(indices[0])
    
    # --- Keyword results ---
    tokens    = query.lower().split()
    scores    = bm25.get_scores(tokens)
    keyword_idx = list(np.argsort(scores)[::-1][:top_k])
    
    # --- Combine both (remove duplicates, keep order) ---
    seen     = set()
    combined = []
    for idx in semantic_idx + keyword_idx:
        if idx not in seen:
            seen.add(idx)
            combined.append(texts[idx])
    
    print(f"Query: '{query}'")
    print(f"\nHybrid results ({len(combined)} total):\n")
    for i, doc in enumerate(combined):
        print(f"{i+1}. {doc}\n")
    
    return combined

# Test it
results = hybrid_search("customers who buy beauty products from north region")

Query: 'customers who buy beauty products from north region'

Hybrid results (10 total):

1. Customer George Reynolds from North purchased Plan (category: Beauty) in quantity 6 for total sales of 1992.72.

2. Customer Christopher Obrien from East purchased Plan (category: Beauty) in quantity 5 for total sales of 1660.6.

3. Customer Angela Watson from North purchased Plan (category: Beauty) in quantity 5 for total sales of 1660.6.

4. Customer Alejandra Walton from North purchased Free (category: Beauty) in quantity 2 for total sales of 732.74.

5. Customer Joshua Leonard from North purchased Free (category: Beauty) in quantity 8 for total sales of 2930.96.

6. Customer Elizabeth Johnson from North purchased Whose (category: Electronics) in quantity 5 for total sales of 1195.15.

7. Customer Timothy Higgins from North purchased Whose (category: Electronics) in quantity 8 for total sales of 1912.24.

8. Customer Gerald Gardner from North purchased Whose (category: Electronics) in quanti

In [ ]:
import json

with open("results.json", "r") as f:
    all_results = json.load(f)

print(f"Loaded {len(all_results)} results")

Loaded 300 results
